In [ ]:
!pip install umap-learn transformers datasets

In [ ]:
import torch
import numpy as np
import umap
import matplotlib.pyplot as plt

from transformers import (
    AutoModelForAudioClassification,
    AutoFeatureExtractor
)

from datasets import load_dataset, Audio,  DatasetDict

In [ ]:
model_or = AutoModelForAudioClassification.from_pretrained("openai/whisper-base")
model_ft = AutoModelForAudioClassification.from_pretrained("VasilisAsim/whisper-finetuned-TESS")

feature_extractor = AutoFeatureExtractor.from_pretrained("VasilisAsim/whisper-finetuned-TESS")

In [ ]:
#Φόρτωση Dataset και speaker eindependent split
def load_tess_speaker_independent():
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))

    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename:
            return "OAF"
        elif "YAF" in filename:
            return "YAF"
        return "unknown"

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")

    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))

    return DatasetDict({"train": train_dataset, "test": test_dataset})

dataset = load_tess_speaker_independent()
test_dataset = dataset["test"]

In [ ]:
def extract_embeddings(model, dataset, num_samples=len(test_dataset)):

    embeddings = []
    labels = []
    model.eval()

    for i in range(num_samples):
      sample = dataset[i]
      audio = sample["audio"]["array"]
      label = sample["emotion"]

      inputs = feature_extractor(audio, sampling_rate=16000, return_tensors="pt")

      with torch.no_grad():
          outputs = model.encoder(inputs["input_features"])

      hidden_states = outputs.last_hidden_state

      pooled_embedding = hidden_states.mean(dim=1)

      embeddings.append(pooled_embedding.squeeze().numpy())

      labels.append(label)

    return np.array(embeddings), np.array(labels)

In [ ]:
emb_or, labels = extract_embeddings(model_or, test_dataset)

emb_ft, _ = extract_embeddings(model_ft, test_dataset)

In [ ]:
umap_model = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=42
)

In [ ]:

emb_pre_2d = umap_model.fit_transform(emb_or)

emb_ft_2d = umap_model.fit_transform(emb_ft)

In [ ]:
from sklearn.preprocessing import LabelEncoder

plt.figure(figsize=(10,8))

le = LabelEncoder()
numerical_labels = le.fit_transform(labels)

scatter = plt.scatter(
    emb_pre_2d[:,0],
    emb_pre_2d[:,1],
    c=numerical_labels,
    cmap="tab10"
)

plt.colorbar(scatter)

plt.title("UMAP - Pretrained Whisper")
plt.xlabel("UMAP Dimension 1")
plt.ylabel("UMAP Dimension 2")
plt.show()

In [ ]:
emotion_names = le.classes_

plt.figure(figsize=(10,8))

scatter = plt.scatter(
    emb_ft_2d[:,0],
    emb_ft_2d[:,1],
    c=le.transform(labels),
    cmap="tab10"
)

handles, _ = scatter.legend_elements()

plt.legend(
    handles,
    emotion_names,
    title="Emotion",
    bbox_to_anchor=(1.05,1),
    loc="upper left"
)

plt.title("UMAP - Finetuned Whisper")
plt.xlabel("UMAP Dimension 1")
plt.ylabel("UMAP Dimension 2")

plt.show()